# Ensemble Methods Interactive Exploration

This notebook imports the project implementations directly. It is a compact defense aid for changing model complexity and unsupervised parameters without duplicating algorithm code. The default execution is deterministic and uses the Breast Cancer Wisconsin dataset.

In [2]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from ipywidgets import Dropdown, FloatSlider, IntSlider, Output, VBox
from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split

from src.metrics.evaluation import classification_metrics
from src.trees import AdaBoostClassifier, DecisionTree, RandomForestClassifier
from src.unsupervised import DBSCAN, KMeans, PCA

SEED = 42

ModuleNotFoundError: No module named 'matplotlib'

## Supervised model controls

Choose a dataset and model, then adjust estimator count or depth. Running this cell in Jupyter displays live metrics from the custom implementations.

In [3]:
dataset_control = Dropdown(options=['breast_cancer', 'iris'], value='breast_cancer', description='Dataset')
model_control = Dropdown(options=['Decision Tree', 'AdaBoost', 'Random Forest'], value='AdaBoost', description='Model')
estimators_control = IntSlider(min=1, max=100, step=1, value=20, description='Estimators')
depth_control = IntSlider(min=0, max=10, step=1, value=3, description='Depth')
supervised_output = Output()

def supervised_demo(*_):
    with supervised_output:
        supervised_output.clear_output(wait=True)
        bunch = load_breast_cancer() if dataset_control.value == 'breast_cancer' else load_iris()
        X_train, X_test, y_train, y_test = train_test_split(
            bunch.data, bunch.target, test_size=0.2, stratify=bunch.target, random_state=SEED
        )
        if model_control.value == 'Decision Tree':
            model = DecisionTree(max_depth=depth_control.value, random_state=SEED)
        elif model_control.value == 'AdaBoost':
            model = AdaBoostClassifier(n_estimators=estimators_control.value, random_state=SEED)
        else:
            model = RandomForestClassifier(
                n_estimators=estimators_control.value,
                max_depth=depth_control.value,
                random_state=SEED,
            )
        model.fit(X_train, y_train)
        metrics = classification_metrics(
            y_test, model.predict(X_test), model.predict_proba(X_test), np.unique(bunch.target)
        )
        print(type(model).__name__, metrics)

for control in (dataset_control, model_control, estimators_control, depth_control):
    control.observe(supervised_demo, names='value')
display(VBox([dataset_control, model_control, estimators_control, depth_control, supervised_output]))
supervised_demo()

NameError: name 'Dropdown' is not defined

## PCA and clustering controls

The controls below vary PCA components, K-Means *k*, and DBSCAN density parameters. The scatter view always uses the first two fitted PCs; clustering itself uses the selected PCA representation.

In [4]:
pca_control = IntSlider(min=2, max=4, value=2, description='PCs')
k_control = IntSlider(min=1, max=10, value=3, description='K')
eps_control = FloatSlider(min=0.1, max=3.0, step=0.1, value=0.8, description='Epsilon')
min_samples_control = IntSlider(min=2, max=20, value=5, description='Min pts')
cluster_control = Dropdown(options=['K-Means', 'DBSCAN'], value='K-Means', description='Cluster')
cluster_output = Output()

def clustering_demo(*_):
    with cluster_output:
        cluster_output.clear_output(wait=True)
        bunch = load_iris()
        scaled = (bunch.data - bunch.data.mean(axis=0)) / bunch.data.std(axis=0)
        embedding = PCA(pca_control.value).fit_transform(scaled)
        if cluster_control.value == 'K-Means':
            labels = KMeans(k_control.value, random_state=SEED).fit(embedding).labels_
        else:
            labels = DBSCAN(eps_control.value, min_samples_control.value).fit(embedding).labels_
        plt.figure(figsize=(6, 4))
        plt.scatter(embedding[:, 0], embedding[:, 1], c=labels, s=24, cmap='tab10')
        plt.xlabel('PC 1')
        plt.ylabel('PC 2')
        plt.title(f'{cluster_control.value} on Iris PCA representation')
        plt.tight_layout()
        plt.show()
        print('Cluster labels and counts:', np.unique(labels, return_counts=True))

for control in (pca_control, k_control, eps_control, min_samples_control, cluster_control):
    control.observe(clustering_demo, names='value')
display(VBox([pca_control, k_control, eps_control, min_samples_control, cluster_control, cluster_output]))
clustering_demo()

NameError: name 'IntSlider' is not defined

## Interpretation guardrail

PCA preserves directions of greatest linear variance, not class separation. K-Means assumes compact Euclidean clusters, whereas DBSCAN identifies dense connected regions and explicit noise. Class labels are therefore used only for external comparison; clusters are not expected to reproduce classes exactly.